# Day 1: Basic Data Cleaning and EDA
**Dataset:** Breast Cancer Diagnosis  
**Goal:** Understand the dataset, clean it, make 4 charts, and create 2 new features.


## 1. Import libraries
`pandas` reads and cleans data. `matplotlib` and `seaborn` make charts.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load and inspect the data
Each row is one sample. `diagnosis` is **M** (malignant) or **B** (benign).

In [ ]:
df = pd.read_csv('data.csv')
print('Rows and columns:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Check missing values
`isnull().sum()` counts missing entries in each column. Our file has one completely empty extra column, `Unnamed: 32`, so we remove it.

In [ ]:
print(df.isnull().sum())

In [ ]:
df = df.drop(columns=['Unnamed: 32'])
print('Shape after dropping empty column:', df.shape)
print('Remaining missing values:', df.isnull().sum().sum())

## 4. Check duplicates
A duplicate is a row that appears more than once. `drop_duplicates()` removes identical copies if any exist.

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates()
print('Rows after cleaning:', len(df))

## 5. Check possible outliers
Outliers are unusually high or low values. We use the **IQR rule** on `radius_mean`. We only *flag* them, because unusual medical measurements may be real and should not automatically be deleted.

In [ ]:
Q1 = df['radius_mean'].quantile(0.25)
Q3 = df['radius_mean'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['radius_mean'] < lower) | (df['radius_mean'] > upper)]
print('Possible radius outliers:', len(outliers))
print('We keep these rows because they might be valid measurements.')

## 6. Exploratory Data Analysis (EDA)
EDA means using simple charts to understand patterns in the data.

### Chart 1: Count plot
How many samples have diagnosis B versus M?

In [ ]:
sns.countplot(data=df, x='diagnosis', color='steelblue')
plt.title('Count of Benign (B) and Malignant (M) Samples')
plt.xlabel('Diagnosis')
plt.ylabel('Number of samples')
plt.show()

### Chart 2: Histogram
How is `radius_mean` distributed?

In [ ]:
plt.hist(df['radius_mean'], bins=20, edgecolor='black')
plt.title('Distribution of Mean Radius')
plt.xlabel('Mean radius')
plt.ylabel('Number of samples')
plt.show()

### Chart 3: Boxplot
Compare mean radius for benign and malignant groups. The line inside each box is the median.

In [ ]:
sns.boxplot(data=df, x='diagnosis', y='radius_mean', color='lightblue')
plt.title('Mean Radius by Diagnosis')
plt.xlabel('Diagnosis')
plt.ylabel('Mean radius')
plt.show()

### Chart 4: Correlation heatmap
Correlation shows whether two numerical measurements tend to change together. Values close to **+1** show a strong positive relationship.

In [ ]:
columns = ['radius_mean', 'perimeter_mean', 'area_mean', 'texture_mean']
sns.heatmap(df[columns].corr(), annot=True, cmap='Blues')
plt.title('Correlation Between Four Measurements')
plt.show()

## 7. Feature engineering
A **feature** is a column that could help a future model. We create two simple new columns by comparing worst measurements with mean measurements.

In [ ]:
df['radius_difference'] = df['radius_worst'] - df['radius_mean']
df['area_difference'] = df['area_worst'] - df['area_mean']
df[['radius_mean', 'radius_worst', 'radius_difference', 'area_mean', 'area_worst', 'area_difference']].head()

## 8. Check group averages
This table helps us describe our findings in simple words.

In [ ]:
print(df['diagnosis'].value_counts())
print(df.groupby('diagnosis')[['radius_mean', 'area_mean', 'texture_mean']].mean().round(2))

## 9. Five key insights

1. The dataset has **569 samples** and originally **33 columns**.
2. The `Unnamed: 32` column is completely empty, so we removed it. There are **no other missing values** and **no duplicate rows**.
3. There are **357 benign (B)** samples and **212 malignant (M)** samples, so benign samples are more common.
4. The malignant group has a **higher average mean radius** (about **17.46**) than the benign group (about **12.15**). These group averages overlap and do not diagnose individual cases.
5. `radius_mean` and `area_mean` are **strongly positively correlated** (about **0.99**). The IQR method flagged **14 possible mean-radius outliers**, which we kept rather than assuming they were errors.

**Conclusion:** We checked data quality, inspected unusual values, created four charts, and engineered two new features. We did not train a predictive model.

In [ ]:
# df.to_csv('cleaned_data.csv', index=False)